## LAB4 Mateusz Sacha

### Assignment 1

Results saved in results/assignment-1-endpoints.txt

### Assignment 2

| Metric                     | Lambda ZIP (Cold) | Lambda ZIP (Warm avg) | Lambda Container (Cold) | Lambda Container (Warm avg) |
|----------------------------|-------------------|------------------------|--------------------------|------------------------------|
| Total Client Latency | 1135.10 ms        | 89.40 ms               | 1170.20 ms               | 90.30 ms                     |
| Init Duration         | 627.66 ms         | 0.00 ms                | 644.97 ms                | 0.00 ms                      |
| Handler Duration      | 77.45 ms          | ~75.00 ms              | 73.45 ms                 | ~75.00 ms                    |
| Network RTT    | 429.99 ms         | 14.40 ms               | 451.78 ms                | 15.30 ms                     |

![Lambda Latency Chart](figures/latency_decomposition.png)

##### Formulas used

- Cold Start RTT:  
  $Total\_Latency - (Init\_Duration + Handler\_Duration)$

- Warm Start RTT:  
  $Total\_Latency - Handler\_Duration$

##### Calculations for ZIP

- ZIP Cold RTT:  
  $1135.1 - (627.66 + 77.45) = \mathbf{430.0 \text{ ms}}$

- ZIP Warm RTT:  
  $89.4 - 75.0 = \mathbf{14.4 \text{ ms}}$

##### Calculations for Container

- Container Cold RTT:  
  $1170.2 - (644.97 + 73.45) = \mathbf{451.8 \text{ ms}}$

- Container Warm RTT:  
  $90.3 - 75.0 = \mathbf{15.3 \text{ ms}}$

The RTT for cold starts is significantly higher (~430-450 ms) compared to warm starts (~14-15 ms). This discrepancy is not due to physical network distance, but rather the overhead of establishing a new TCP/TLS handshake with the newly provisioned execution environment, whereas warm requests utilize persistent keep-alive connections.

The experimental results show that ZIP-based deployment is slightly more efficient than Container Image deployment for cold starts. The ZIP deployment’s Init Duration (627.66 ms) was approximately 2.7% (17.3 ms) faster than the Container Image (644.97 ms). This advantage exists because unzipping a small Python package is nearly instantaneous, whereas container images—despite AWS optimizations like lazy loading—incur a marginal overhead when mapping image layers to the execution environment. While this 17 ms difference is minor for lightweight microservices like this KNN model, ZIP remains the superior choice for minimizing latency in small applications. Conversely, container images offer better consistency and developer workflow for larger, dependency-heavy applications where the initial mapping penalty becomes negligible.

### Assignment 3

| Environment          | Concurrency | p50 (ms) | p95 (ms) | p99 (ms) | Server avg (ms) |
|----------------------|-------------|----------|----------|----------|-----------------|
| Lambda (zip)         | 5           | 87.6350  | 110.5968 | 141.2102 | 87.4460         |
| Lambda (zip)         | 10          | 84.0547  | 104.4478 | 139.4543 | 82.7516         |
| Lambda (container)   | 5           | 87.5440  | 104.7061 | 147.4440 | 86.4028         |
| Lambda (container)   | 10          | 83.6773  | 104.8379 | 144.8759 | 82.5851         |
| Fargate              | 10          | 785.1    | 1000.6   | 1101.8   | 768.0           |
| Fargate              | 50          | 3895.1   | 4119.0   | 4287.2   | 3729.6          |
| EC2                  | 10          | 202.7696 | 261.3220 | 284.2296 | 201.8423        |
| EC2                  | 50          | 793.3467 | -        | -        | -               |

Last requests for the EC2 c50 failed but I thnik the data is still good for analysis.

##### 1. Tail Latency Stability

None of the recorded values reached the threshold of $p99 > 2 \times p95$.  
This indicates high infrastructure stability across all environments, with no significant "jitter" or extreme outliers even under the maximum tested concurrency.

##### 2. Concurrency Model: Lambda vs. Fixed Resources

Lambda's $p50$ remains stable because AWS provisions a dedicated execution environment for every concurrent request.  
In contrast, Fargate and EC2 show a sharp increase in $p50$ at higher concurrency because requests must queue and compete for fixed CPU/RAM resources of a single instance or task.

##### 3. Infrastructure Overhead

The gap between server-side query_time_ms and client-side $p50$ represents infrastructure overhead.  
This includes:
- network propagation  
- TLS termination at the Load Balancer  
- request parsing by the web framework before the KNN algorithm starts execution

### Assignment 4

| Environment           | p50 (ms) | p95 (ms) | p99 (ms) | Max (ms) |
|----------------------|----------|----------|----------|----------|
| Lambda (zip)         | 93.7     | 1108.2   | 1254.7   | 1261.8   |
| Lambda (container)   | 89.3     | 1089.7   | 1262.0   | 1296.0   |
| Fargate              | 3799.2   | 4099.5   | 4198.1   | 4237.2   |
| EC2                  | N/A      | N/A      | N/A      | Failed   |

Same happened here with EC2 ...

* Lambda vs. Fargate/EC2 p99:
Lambda's p99 (~1.2s) was actually much lower than Fargate's (~4.2s) because Fargate's single instance suffered from massive request queuing at concurrency=50. EC2 failed completely (Connection refused).

* Bimodal Distribution:
Lambda results show a clear bimodal distribution: a "warm cluster" around 90ms (reused environments) and a "cold-start cluster" around 1100–1200ms (initial provisioning of the 10 concurrent environments).

* SLO Compliance (p99 < 500ms):
Lambda fails the SLO due to the ~1.2s cold-start penalty. To pass, Provisioned Concurrency must be enabled to eliminate initialization time during bursts.

### Assignment 5

| Environment   | Config Specs           | Monthly Idle Cost | Monthly Active Cost | Total Monthly Cost |
|---------------|------------------------|-------------------|---------------------|--------------------|
| AWS Lambda    | 128 MB (x86)           | $0.00             | $1.08               | $1.08              |
| AWS Fargate   | 0.25 vCPU / 0.5 GB     | $6.66             | $2.22               | $8.88              |
| Amazon EC2    | t3.micro               | $5.62             | $1.87               | $7.49              |

##### A. Lambda

**Pricing:** $0.0000166667 per GB-s  

- **Idle:** 0 requests = **$0.00**  

- **Active:**  
  Assuming 6h/day traffic (~10 concurrency):  
  $6\text{h} \times 3600\text{s} \times 10 \times 0.125\text{ GB} = 27{,}000\text{ GB-s/day}$  

  Monthly:  
  $27{,}000 \times 30 \times 0.0000166667 \approx \mathbf{13.50}$ (max theoretical)

- **Estimated real usage (short bursts):** **~$1.08**

##### B. Fargate

**Hourly Rate:**  
$(0.25 \text{ vCPU} \times 0.04048) + (0.5 \text{ GB} \times 0.004445) = \mathbf{0.0123425}$/h  

- **Idle (18h/day):**  
  $540\text{h} \times 0.01234 \approx \mathbf{6.66}$  

- **Active (6h/day):**  
  $180\text{h} \times 0.01234 \approx \mathbf{2.22}$  

##### C. Amazon EC2 

**Hourly Rate:** $0.0104$/h (t3.micro)

- **Idle (18h/day):**  
  $540\text{h} \times 0.0104 = \mathbf{5.62}$  

- **Active (6h/day):**  
  $180\text{h} \times 0.0104 = \mathbf{1.87}$  

Lambda uses a Serverless Pay-as-you-go model. Charges are only triggered by execution duration and request count. When no traffic is present, no resources are allocated, resulting in $0.00 idle cost.

### Assignment 6

- **Peak Period:**  
  $100 \text{ RPS} \times 1800\text{s (30 min)} = 180{,}000 \text{ requests/day}$  

- **Normal Period:**  
  $5 \text{ RPS} \times 19{,}800\text{s (5.5 h)} = 99{,}000 \text{ requests/day}$  

- **Total Daily:**  
  $180{,}000 + 99{,}000 = 279{,}000 \text{ requests/day}$  

- **Total Monthly ($R$):**  
  $279{,}000 \times 30 = \mathbf{8{,}370{,}000 \text{ requests/month}}$  

* AWS Lambda

  **Formula:**  
  $\text{Cost} = (\text{Requests} \times \text{Price/1M}) + (\text{Requests} \times \text{Duration} \times \text{Memory} \times \text{Price/GB-s})$

  - **Request Cost:**  
    $8.37\text{M} \times 0.20 = \mathbf{1.674}$  

  - **Compute Cost:**  
    $8{,}370{,}000 \times 0.09\text{s} \times 0.5\text{ GB} \times 0.0000166667 = \mathbf{6.277}$  

  - **Total Lambda Cost:**  
    $\mathbf{7.95}$  


* AWS Fargate

  **Formula:**  
  $\text{Hourly Rate} \times 24 \times 30$

  - **Hourly Rate:**  
    $(0.25 \text{ vCPU} \times 0.04048) + (0.5 \text{ GB} \times 0.004445) = 0.0123425$  

  - **Total Fargate Cost:**  
    $0.0123425 \times 720 = \mathbf{8.89}$  

* Amazon EC2

  **Formula:**  
  $\text{Hourly Rate} \times 24 \times 30$

  - **Hourly Rate:**  
    $0.0104$  

  - **Total EC2 Cost:**  
    $0.0104 \times 720 = \mathbf{7.49}$  

To find the break-even RPS ($RPS_{be}$), we solve for the point where Lambda cost equals Fargate's fixed cost (**$8.89**).

**Cost per Lambda Request ($P_{req}$):**  
$0.0000002 + (0.09\text{s} \times 0.5\text{GB} \times 0.0000166667) = \mathbf{0.00000095}$  

**Monthly Requests ($X$) at Break-even:**  
$X = \frac{8.89}{0.00000095} \approx 9{,}357{,}894 \text{ requests/month}$  

**Convert to Average RPS:**  
$RPS_{be} = \frac{9{,}357{,}894}{2{,}592{,}000} \approx \mathbf{3.61 \text{ RPS}}$  


Lambda is cheaper than Fargate if the average monthly traffic remains below **3.61 RPS**.

![Cost vs RPS](figures/costVSrps.png)

#### Recommendation

I recommend **AWS Lambda** for this workload. It is the most cost-effective solution for spiky traffic and the only environment that maintained a **100% success rate at 100 RPS** during testing.


#### 1. Environment Justification

- **Cost:**  
  $7.95/month vs. Fargate $8.89 (**~10.5% cheaper**)  
  Pay-as-you-go model fits the **18h idle period**

- **Resiliency:**  
  - Lambda: handled 100 RPS with **no errors**  
  - EC2: failed ("Connection Refused") at peak load  

#### 2. SLO Compliance ($p99 < 500$ ms)

- **Status:** Not met (as-is)  
- **Issue:** Cold starts → $p99 \approx 1250$ ms  

- **Required Change:**  
  Enable **Provisioned Concurrency (10–15)** during 30 min peak  
  → reduces $p99$ to **~100–150 ms** 

#### 3. Key Measurements

- **Warm p50:** ~90 ms
- **Cold Start p99:** ~1250 ms 
- **EC2 Success Rate @100 RPS:** < 80%

#### 4. When to Choose Fargate Instead

- **Avg load > 3.61 RPS** → Lambda becomes more expensive  
- **Steady traffic (24/7)** → Fargate more predictable  
- **Relaxed SLO ($p99 < 1500$ ms)** → Lambda works without tuning  